In [ ]:
# 0,1: 00 for psi_1, 01 for psi_2 ...
# 2,3: For states
# 4,5: Ancillary qubits

In [ ]:
from qiskit import QuantumCircuit
from numpy import sqrt

def circuit_init():
    qc = QuantumCircuit(6)
    return qc

def PREP(qc):
    desired_vector = [
        1/sqrt(3), 0, 0, 0, 
        1/(4*sqrt(3)), 1/4, 1/4, sqrt(3)/4, 
        1/(4*sqrt(3)), -1/4, -1/4, sqrt(3)/4, 
        0, 0, 0, 0
    ]
    qc.initialize(desired_vector, [3,2,1,0])
    return qc

In [ ]:
from qiskit.circuit.library.standard_gates import RXGate,RYGate, RZGate
from qiskit.circuit.library import MCXGate


def UU(qc, wires, params):
    qc.append(U2Gate(params[0],params[1]),[wires[0]])
    qc.append(U2Gate(params[2],params[3]),[wires[1]])


def RR_Z(qc, wires, params):
    qc.append(RZGate(params[0]),[wires[0]])
    qc.append(RZGate(params[1]),[wires[1]])

def CUU(qc,wires, params, state='1'):
    qc.append(U2Gate(params[0],params[1]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]
    qc.append(U2Gate(params[2],params[3]).control(1,ctrl_state = state[::-1]), [wires[0], wires[2]])  # Control: wires[0] Target: wires[2]
    return qc

def CRR_Z(qc,wires, params, state='1'):
    qc.append(RZGate(params[0]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]
    qc.append(RZGate(params[1]).control(1,ctrl_state = state[::-1]), [wires[0], wires[2]])  # Control: wires[0] Target: wires[2]
    return qc

def CR_Y(qc, wires, params, state = '1'):
    qc.append(RYGate(params[0]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]


def CCUU(qc,wires, params, state='11'):
    qc.append(U2Gate(params[0],params[1]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    qc.append(U2Gate(params[2],params[3]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[3]])  # Control: wires[0] wires[1]

def CCRR_Y(qc, wires, params, state = '11'):
    qc.append(RYGate(params[0]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    qc.append(RYGate(params[1]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[3]])  # Control: wires[0] wires[1]
    return qc
    

def CCR_Y(qc, wires, params, state = '11'):
    qc.append(RYGate(params[0]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    return qc

def U_CCR(qc, wires, params):
    qc.append(RYGate(params[0]).control(2, ctrl_state = '00'), [wires[0], wires[1],wires[2]])
    qc.append(RYGate(params[1]).control(2, ctrl_state = '10'), [wires[0], wires[1],wires[2]])
    qc.append(RYGate(params[2]).control(2, ctrl_state = '01'), [wires[0], wires[1],wires[2]])
    qc.append(RYGate(params[3]).control(2, ctrl_state = '11'), [wires[0], wires[1],wires[2]])

def U_CCcR(qc,wires, params):
    qc.append(RYGate(params[0]).control(3, ctrl_state = '001'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[1]).control(3, ctrl_state = '011'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[2]).control(3, ctrl_state = '101'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[3]).control(3, ctrl_state = '111'[::-1]), [wires[0], wires[1],wires[2],wires[3]])

def CCX(qc, wires, state = '11'):
    mcx_gate = MCXGate(num_ctrl_qubits=2,ctrl_state=state[::-1])
    qc.append(mcx_gate, [wires[0], wires[1],wires[2]])
    return qc

def CCCX(qc, wires, state = '111'):
    mcx_gate = MCXGate(num_ctrl_qubits=3,ctrl_state=state[::-1])
    qc.append(mcx_gate, [wires[0], wires[1],wires[2], wires[3]])
    return qc

In [ ]:
from qiskit.circuit import ParameterVector
from qiskit.circuit.library.standard_gates import RYGate
from qiskit.circuit.library import U2Gate
from numpy import pi

# U2Gate 
def Ansatz(qc, wires):

    params = ParameterVector('x', 63)
    
    #Big made by HEA 
    UU(qc,wires[0:2],params[0:4])
    qc.cx(wires[1],wires[0])
    RR_Z(qc,wires[0:2],params[4:6])
    qc.cx(wires[0],wires[1])
    qc.ry(params[6],wires[1])
    qc.cx(wires[1],wires[0])
    UU(qc,wires[0:2],params[7:11])

    qc.barrier()
    #Module 1
    U_CCR(qc,wires[0:3],params[11:15])

    CUU(qc,[wires[2],wires[0],wires[1]],params[15:19],'0')
    CCX(qc,[wires[1],wires[2], wires[0]],'10')
    CRR_Z(qc,[wires[2],wires[0],wires[1]],params[19:21],'0')
    CCX(qc,[wires[0],wires[2], wires[1]],'10')
    CR_Y(qc,[wires[2], wires[1]],[params[21]],'0')
    CCX(qc,[wires[1],wires[2], wires[0]],'10')
    CUU(qc,[wires[2],wires[0],wires[1]],params[22:26],'0')
    
    CUU(qc,[wires[2],wires[0],wires[1]],params[26:30],'1')
    CCX(qc,[wires[1],wires[2], wires[0]],'11')
    CRR_Z(qc,[wires[2],wires[0],wires[1]],params[30:32],'1')
    CCX(qc,[wires[0],wires[2], wires[1]],'11')
    CR_Y(qc,[wires[2], wires[1]],[params[32]],'1')
    CCX(qc,[wires[1],wires[2], wires[0]],'11')
    CUU(qc,[wires[2],wires[0],wires[1]],params[33:37],'1')

    qc.barrier()
    #Module 2
    U_CCcR(qc, wires[0:4],params[37:41])
    CCUU(qc,[wires[2],wires[3],wires[0],wires[1]],params[41:45],'10')
    CCCX(qc,[wires[1],wires[2],wires[3],wires[0]],'110')
    CCRR_Y(qc,[wires[2],wires[3],wires[0],wires[1]], params[45:47], '10')
    CCCX(qc,[wires[0],wires[2],wires[3],wires[1]],'110')
    CCR_Y(qc,[wires[2],wires[3],wires[1]],[params[47]],'10')
    CCCX(qc,[wires[1],wires[2],wires[3],wires[0]],'110')
    CCUU(qc,[wires[2],wires[3],wires[0],wires[1]],params[48:52],'10')
    
    CCUU(qc,[wires[2],wires[3],wires[0],wires[1]],params[52:56],'11')
    CCCX(qc,[wires[1],wires[2],wires[3],wires[0]],'111')
    CCRR_Y(qc,[wires[2],wires[3],wires[0],wires[1]], params[56:58], '11')
    CCCX(qc,[wires[0],wires[2],wires[3],wires[1]],'111')
    CCR_Y(qc,[wires[2],wires[3],wires[1]],[params[58]],'11')
    CCCX(qc,[wires[1],wires[2],wires[3],wires[0]],'111')
    CCUU(qc,[wires[2],wires[3],wires[0],wires[1]],params[59:63],'11')
    
    
    qc.measure_all()
    return qc

In [ ]:
from numpy.random import rand
num_params=63
qc = circuit_init()
PREP(qc)
Ansatz(qc, [2,3,4,5])
qc_reversed=qc.reverse_bits()
qc.draw(output='mpl', style = 'clifford') 

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel='ibm_quantum')
#backend = service.least_busy(min_num_qubits=127)
backend = service.backend("ibm_strasbourg")
print(backend)

pm = generate_preset_pass_manager(optimization_level=3,backend=backend)


candidate_circuit = pm.run(qc_reversed)
candidate_circuit.draw('mpl', fold=False, scale=0.1, idle_wires=False)

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# Define the factor (I + Z)/2 = |0><0|
o_term = SparsePauliOp.from_list([("I", 0.5), ("Z", 0.5)])

# Define the factor (I - Z)/2 = |1><1|
l_term = SparsePauliOp.from_list([("I", 0.5), ("Z", -0.5)])

I = SparsePauliOp.from_list([("I", 1)])

# Use tensor products to create (I + Z)/2 ⊗ (I + Z)/2 ⊗ (I + Z)/2 ⊗ (I + Z)/2
Ob1= o_term.tensor(o_term).tensor(I).tensor(I).tensor(o_term).tensor(o_term)

# Check if b0b1 = 01, Check if b3b4 = 10
Ob2= o_term.tensor(l_term).tensor(I).tensor(I).tensor(l_term).tensor(o_term)

# Check if b0b1 = 10, Check if b3b4 = 11
Ob3= l_term.tensor(o_term).tensor(I).tensor(I).tensor(l_term).tensor(l_term)

cost_hamiltonian = Ob1 + Ob2 + Ob3
cost_hamiltonian=cost_hamiltonian.apply_layout(candidate_circuit.layout)

In [ ]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # Prepare the job input with the ansatz, Hamiltonian, and parameters
    pub = (ansatz, hamiltonian, params)
    job = estimator.run([pub])

    # Retrieve the result
    results = job.result()[0]

    # Extract the cost (expectation value)
    cost = results.data.evs
    print('cost is', cost)

    # Append cost to the global success probability list and return it
    success_probability.append(cost)
    return 1/cost

In [ ]:
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from scipy.optimize import minimize

init_params = rand(num_params)
success_probability = [] # Global variable

with Session(backend=backend) as session:

    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000
    estimator.options.max_execution_time = 4500

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-1,
    )
    print(result)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(success_probability, label="Sucess Probability")
plt.xlabel('Iterations')
plt.ylabel('Sucess Probability')
plt.legend()
plt.show()

In [ ]:
import openpyxl

# create a new workbook
workbook = openpyxl.Workbook()

# select the active worksheet
worksheet = workbook.active

# loop through the confidences array and write values to the worksheet
for i in range(len(success_probability)):
    worksheet.cell(row=i+1, column=1, value=float(success_probability[i]))

# save the workbook to a file
workbook.save('ME Yordan+Exact DT states tol01 (ibm_strasbourg).xlsx')